In [ ]:
#Install necessary library
!pip install nltk vaderSentiment wordcloud

In [ ]:
#import and load necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
#load dataset into model
df = pd.read_csv("/content/vaccination_all_tweets.csv")
df.head(10) #print the first few row

,id,user_name,user_location,user_description,user_created,user_followers,user_friends,user_favourites,user_verified,date,text,hashtags,source,retweets,favorites,is_retweet
0,1340539111971516416,Rachel Roh,"La Crescenta-Montrose, CA",Aggregator of Asian American news; scanning di...,2009-04-08 17:52:46,405,1692,3247,False,2020-12-20 06:06:44,Same folks said daikon paste could treat a cyt...,['PfizerBioNTech'],Twitter for Android,0,0,False
1,1338158543359250433,Albert Fong,"San Francisco, CA","Marketing dude, tech geek, heavy metal & '80s ...",2009-09-21 15:27:30,834,666,178,False,2020-12-13 16:27:13,While the world has been on the wrong side of ...,NaN,Twitter Web App,1,1,False
2,1337858199140118533,eli🇱🇹🇪🇺👌,Your Bed,"heil, hydra 🖐☺",2020-06-25 23:30:28,10,88,155,False,2020-12-12 20:33:45,#coronavirus #SputnikV #AstraZeneca #PfizerBio...,"['coronavirus', 'SputnikV', 'AstraZeneca', 'Pf...",Twitter for Android,0,0,False
3,1337855739918835717,Charles Adler,"Vancouver, BC - Canada","Hosting ""CharlesAdlerTonight"" Global News Radi...",2008-09-10 11:28:53,49165,3933,21853,True,2020-12-12 20:23:59,"Facts are immutable, Senator, even when you're...",NaN,Twitter Web App,446,2129,False
4,1337854064604966912,Citizen News Channel,NaN,Citizen News Channel bringing you an alternati...,2020-04-23 17:58:42,152,580,1473,False,2020-12-12 20:17:19,Explain to me again why we need a vaccine @Bor...,"['whereareallthesickpeople', 'PfizerBioNTech']",Twitter for iPhone,0,0,False
5,1337852648389832708,Dee,"Birmingham, England","Gastroenterology trainee, Clinical Research Fe...",2020-01-26 21:43:12,105,108,106,False,2020-12-12 20:11:42,Does anyone have any useful advice/guidance fo...,NaN,Twitter for iPhone,0,0,False
6,1337851215875608579,Gunther Fehlinger,"Austria, Ukraine and Kosovo",End North Stream 2 now - the pipeline of corru...,2013-06-10 17:49:22,2731,5001,69344,False,2020-12-12 20:06:00,it is a bit sad to claim the fame for success ...,['vaccination'],Twitter Web App,0,4,False
7,1337850832256176136,Dr.Krutika Kuppalli,NaN,"ID, Global Health, VHF, Pandemic Prep, Emergin...",2019-03-25 04:14:29,21924,593,7815,True,2020-12-12 20:04:29,There have not been many bright days in 2020 b...,"['BidenHarris', 'Election2020']",Twitter for iPhone,2,22,False
8,1337850023531347969,Erin Despas,NaN,Designing&selling on Teespring. Like 90s Disne...,2009-10-30 17:53:54,887,1515,9639,False,2020-12-12 20:01:16,Covid vaccine; You getting it?\n\n #CovidVacci...,"['CovidVaccine', 'covid19', 'PfizerBioNTech', ...",Twitter Web App,2,1,False
9,1337842295857623042,Ch.Amjad Ali,Islamabad,#ProudPakistani #LovePakArmy #PMIK @insafiansp...,2012-11-12 04:18:12,671,2368,20469,False,2020-12-12 19:30:33,#CovidVaccine \n\nStates will start getting #C...,"['CovidVaccine', 'COVID19Vaccine', 'US', 'paku...",Twitter Web App,0,0,False


In [ ]:
#print dataset info
df.info()
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 228207 entries, 0 to 228206
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   id                228207 non-null  int64 
 1   user_name         228205 non-null  object
 2   user_location     161296 non-null  object
 3   user_description  211184 non-null  object
 4   user_created      228207 non-null  object
 5   user_followers    228207 non-null  int64 
 6   user_friends      228207 non-null  int64 
 7   user_favourites   228207 non-null  int64 
 8   user_verified     228207 non-null  bool  
 9   date              228207 non-null  object
 10  text              228207 non-null  object
 11  hashtags          178504 non-null  object
 12  source            228088 non-null  object
 13  retweets          228207 non-null  int64 
 14  favorites         228207 non-null  int64 
 15  is_retweet        228207 non-null  bool  
dtypes: bool(2), int64(6), object(8)
memory

In [ ]:
df.columns

Index(['id', 'user_name', 'user_location', 'user_description', 'user_created',
       'user_followers', 'user_friends', 'user_favourites', 'user_verified',
       'date', 'text', 'hashtags', 'source', 'retweets', 'favorites',
       'is_retweet'],
      dtype='object')

In [ ]:
#keep necessary attributes columns
df = df[['text', 'hashtags', 'date', 'retweets', 'favorites']]
df.head()
print (df.head(10))

                                                text  \
0  Same folks said daikon paste could treat a cyt...   
1  While the world has been on the wrong side of ...   
2  #coronavirus #SputnikV #AstraZeneca #PfizerBio...   
3  Facts are immutable, Senator, even when you're...   
4  Explain to me again why we need a vaccine @Bor...   
5  Does anyone have any useful advice/guidance fo...   
6  it is a bit sad to claim the fame for success ...   
7  There have not been many bright days in 2020 b...   
8  Covid vaccine; You getting it?\n\n #CovidVacci...   
9  #CovidVaccine \n\nStates will start getting #C...   

                                            hashtags                 date  \
0                                 ['PfizerBioNTech']  2020-12-20 06:06:44   
1                                                NaN  2020-12-13 16:27:13   
2  ['coronavirus', 'SputnikV', 'AstraZeneca', 'Pf...  2020-12-12 20:33:45   
3                                                NaN  2020-12-12 20:23:59  

In [ ]:
#clean the 'text' column
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)     # remove URLs
    text = re.sub(r"@\w+", "", text)        # remove mentions
    text = re.sub(r"#", "", text)           # remove hashtag symbol
    text = re.sub(r"[^a-z\s]", "", text)    # remove punctuation & numbers
    tokens = nltk.word_tokenize(text)       # tokenization
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words] #lemmatization
    return " ".join(tokens)

df['clean_text'] = df['text'].apply(clean_text)
df.head()

print ("DATASET", df.head(10))
print ("\nClean task done.")

DATASET                                                 text  \
0  Same folks said daikon paste could treat a cyt...   
1  While the world has been on the wrong side of ...   
2  #coronavirus #SputnikV #AstraZeneca #PfizerBio...   
3  Facts are immutable, Senator, even when you're...   
4  Explain to me again why we need a vaccine @Bor...   
5  Does anyone have any useful advice/guidance fo...   
6  it is a bit sad to claim the fame for success ...   
7  There have not been many bright days in 2020 b...   
8  Covid vaccine; You getting it?\n\n #CovidVacci...   
9  #CovidVaccine \n\nStates will start getting #C...   

                                            hashtags                 date  \
0                                 ['PfizerBioNTech']  2020-12-20 06:06:44   
1                                                NaN  2020-12-13 16:27:13   
2  ['coronavirus', 'SputnikV', 'AstraZeneca', 'Pf...  2020-12-12 20:33:45   
3                                                NaN  2020-12-12 20

In [ ]:
# Remove empty rows after cleaning
df = df[df['clean_text'].str.strip() != ""]
print("Remaining rows:", df.shape[0])

Remaining rows: 227255


In [ ]:
#save 'text' columns
df[['clean_text']].to_csv("clean_text_covidtweet.csv", index=False)
print("Saved 'clean_text' column to clean_text_output.csv")

Saved 'clean_text' column to clean_text_output.csv


In [ ]:
#TF-IDF for 'hashtags' column
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

def clean_hashtags(tags):
    if pd.isna(tags):
        return ""
    tags = str(tags).lower()
    tags = tags.replace("[", "").replace("]", "").replace("'", "")
    tags = tags.replace(",", " ")
    return tags

df['clean_hashtags'] = df['hashtags'].apply(clean_hashtags)

# Calculate TF-IDF for 'clean_hashtags'
tfidf_hashtags = TfidfVectorizer(max_features=5000)
X_hashtags = tfidf_hashtags.fit_transform(df['clean_hashtags'])

print("TF-IDF matrix shape for hashtags:", X_hashtags.shape)

# Display feature names (vocabulary) for hashtags
print("\nTop 100 Feature Names (words in the hashtag vocabulary):")
print(tfidf_hashtags.get_feature_names_out()[:100])

# Display IDF values for the first 100 hashtag features
print("\nIDF values for the first 100 hashtag features:")
print(tfidf_hashtags.idf_[:100])

# Display a sample of the TF-IDF matrix for the first 5 documents for hashtags
sample_docs_hashtags = df['clean_hashtags'].head(5).tolist()
sample_tfidf_matrix_hashtags = tfidf_hashtags.fit_transform(sample_docs_hashtags)

print("\nTF-IDF matrix for the first 5 documents (hashtag sample):")
sample_df_hashtags = pd.DataFrame(sample_tfidf_matrix_hashtags.toarray(), columns=tfidf_hashtags.get_feature_names_out())
display(sample_df_hashtags)

TF-IDF matrix shape for hashtags: (227255, 5000)

Top 100 Feature Names (words in the hashtag vocabulary):
['100crorevaccination' '100crorevaccinations' '100daysofcode' '12inch'
 '15yearsofmiley' '18to44' '1billion' '1billionvaccination' '1crore'
 '1down1togo' '1st' '1stdose' '1stdosecovid19vaccine' '1stdosedone'
 '1stdosevaccine' '1stjab' '1stmoderna' '1stshot' '1stshotdone'
 '1stvaccine' '2021goals' '23newsbulletin' '24juillet' '24july2021'
 '27thmay2021' '29mar' '2dg' '2dgmedicine' '2dose' '2dosesummer' '2g'
 '2nd' '2nd_dose' '2nddose' '2nddosecompleted' '2nddosedone' '2ndjab'
 '2ndjabdone' '2ndshot' '2ndvaccine' '2of2' '2xgrowthwithampere'
 '2xgrowthwithapere' '3emedose' '3g' '3m' '3rddose' '3rdjab' '3rdshot'
 '3rdwave' '42days' '501yv1' '501yv2' '5g' '5ivelive' '5qs' '6pmprime'
 '7at7' '7pm' '90wordedit' '_क' '_द' '_म' '_र' '_श' '_ह' 'a4' 'aai'
 'aanandayya' 'aap' 'aapdelhihealthbulletin' 'aarogy' 'aarogyasetu'
 'aaronrodgers' 'aatmanirbharbharat' 'aatmnirbharbharat' 'ab' 'abc'
 '

,astrazeneca,coronavirus,covid_19,moderna,pfizerbiontech,sputnikv,whereareallthesickpeople
0,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.428411,0.428411,0.428411,0.428411,0.286912,0.428411,0.000000
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0.000000,0.000000,0.000000,0.000000,0.556451,0.000000,0.830881


In [ ]:
#TF-IDF for 'text' column
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# 'clean_text' column is already prepared in a previous cell, so no need to clean again here.

# Calculate TF-IDF for 'clean_text'
tfidf_text = TfidfVectorizer(max_features=5000)
X_text = tfidf_text.fit_transform(df['clean_text'])

print("TF-IDF matrix shape for text:", X_text.shape)

# Display feature names (vocabulary) for text
print("\nTop 100 Feature Names (words in the text vocabulary):")
print(tfidf_text.get_feature_names_out()[:100])

# Display IDF values for the first 100 text features
print("\nIDF values for the first 100 text features:")
print(tfidf_text.idf_[:100])

# Display a sample of the TF-IDF matrix for the first 5 documents for text
sample_docs_text = df['clean_text'].head(5).tolist()
sample_tfidf_matrix_text = tfidf_text.fit_transform(sample_docs_text)

print("\nTF-IDF matrix for the first 5 documents (text sample):")
sample_df_text = pd.DataFrame(sample_tfidf_matrix_text.toarray(), columns=tfidf_text.get_feature_names_out())
display(sample_df_text)

TF-IDF matrix shape for text: (227255, 5000)

Top 100 Feature Names (words in the text vocabulary):
['aap' 'aarthi' 'aatmanirbharbharat' 'aayog' 'aayug' 'ab' 'abc'
 'abhayahas' 'abhayahasta' 'ability' 'able' 'abo' 'abou' 'abroad'
 'absolute' 'absolutely' 'abt' 'abu' 'abudhabi' 'abuse' 'ac' 'academy'
 'accelerate' 'accelerating' 'accept' 'acceptable' 'acceptance' 'accepted'
 'accepting' 'accepts' 'access' 'accomplished' 'according' 'account'
 'accusation' 'accused' 'ache' 'achieve' 'achieved' 'achievement'
 'achieving' 'aching' 'achy' 'acquire' 'acquired' 'across' 'act' 'acting'
 'action' 'activated' 'active' 'actively' 'activity' 'actor' 'actual'
 'actually' 'acute' 'ad' 'add' 'added' 'adding' 'addition' 'additional'
 'addon' 'address' 'addresshe' 'addresshealth' 'adenovirus' 'adjuvant'
 'admin' 'administer' 'administered' 'administering' 'administers'
 'administration' 'admission' 'admit' 'admits' 'admitted' 'adolescent'
 'adscampaign' 'adult' 'advance' 'advanced' 'advantage' 'adverse

,acknowledge,astrazeneca,biggest,born,coronavirus,could,covid,created,cytokine,daikon,...,sturdy,treat,vaccination,vaccine,weve,whereareallthesickpeople,world,wrong,year,youre
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.344041,0.000000,0.000000,0.344041,0.344041,...,0.000000,0.344041,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.000000,0.000000,0.306413,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.306413,0.000000,0.306413,0.000000,0.306413,0.306413,0.247212,0.000000
2,0.000000,0.320251,0.000000,0.000000,0.320251,0.000000,0.320251,0.320251,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.258376,0.000000,0.000000,0.000000,0.000000,0.258376,0.000000
3,0.316228,0.000000,0.000000,0.316228,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.316228,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.316228
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.398475,0.000000,0.493899,0.000000,0.000000,0.000000,0.000000


In [ ]:
df['date'] = pd.to_datetime(df['date']).dt.date
df.head()
print (df.head(10))

                                                text  \
0  Same folks said daikon paste could treat a cyt...   
1  While the world has been on the wrong side of ...   
2  #coronavirus #SputnikV #AstraZeneca #PfizerBio...   
3  Facts are immutable, Senator, even when you're...   
4  Explain to me again why we need a vaccine @Bor...   
5  Does anyone have any useful advice/guidance fo...   
6  it is a bit sad to claim the fame for success ...   
7  There have not been many bright days in 2020 b...   
8  Covid vaccine; You getting it?\n\n #CovidVacci...   
9  #CovidVaccine \n\nStates will start getting #C...   

                                            hashtags        date  retweets  \
0                                 ['PfizerBioNTech']  2020-12-20         0   
1                                                NaN  2020-12-13         1   
2  ['coronavirus', 'SputnikV', 'AstraZeneca', 'Pf...  2020-12-12         0   
3                                                NaN  2020-12-12       

In [ ]:
# Select the desired columns
compiled_df = df[['date', 'clean_text', 'clean_hashtags', 'retweets', 'favorites']]

# Display the first few rows of the compiled DataFrame
display(compiled_df.head())

# Save the compiled DataFrame to a CSV file
compiled_df.to_csv("compiled_data.csv", index=False)
print("Saved compiled data to compiled_data.csv")

,date,clean_text,clean_hashtags,retweets,favorites
0,2020-12-20,folk said daikon paste could treat cytokine st...,pfizerbiontech,0,0
1,2020-12-13,world wrong side history year hopefully bigges...,,1,1
2,2020-12-12,coronavirus sputnikv astrazeneca pfizerbiontec...,coronavirus sputnikv astrazeneca pfizerbion...,0,0
3,2020-12-12,fact immutable senator even youre ethically st...,,446,2129
4,2020-12-12,explain need vaccine whereareallthesickpeople ...,whereareallthesickpeople pfizerbiontech,0,0


Saved compiled data to compiled_data.csv


In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

df['sentiment_score'] = df['clean_text'].apply(
    lambda x: analyzer.polarity_scores(x)['compound']
)

df[['clean_text', 'sentiment_score']].head()

,clean_text,sentiment_score
0,folk said daikon paste could treat cytokine st...,0.4019
1,world wrong side history year hopefully bigges...,-0.1027
2,coronavirus sputnikv astrazeneca pfizerbiontec...,0.2500
3,fact immutable senator even youre ethically st...,0.0000
4,explain need vaccine whereareallthesickpeople ...,0.0000


In [ ]:
def sentiment_label(score):
    if score >= 0.05:
        return "Positive"
    elif score <= -0.05:
        return "Negative"
    else:
        return "Neutral"

df['sentiment'] = df['sentiment_score'].apply(sentiment_label)
df['sentiment'].value_counts()

,count
sentiment,
Neutral,109295
Positive,84392
Negative,33568


In [ ]:
# Select the desired columns
compiled_df = df[['clean_text', 'sentiment_score', 'sentiment']]

# Display the first few rows of the compiled DataFrame
display(compiled_df.head())

# Save the compiled DataFrame to a CSV file
compiled_df.to_csv("sentiment_data.csv", index=False)
print("Saved compiled data to sentiment_data.csv")

,clean_text,sentiment_score,sentiment
0,folk said daikon paste could treat cytokine st...,0.4019,Positive
1,world wrong side history year hopefully bigges...,-0.1027,Negative
2,coronavirus sputnikv astrazeneca pfizerbiontec...,0.2500,Positive
3,fact immutable senator even youre ethically st...,0.0000,Neutral
4,explain need vaccine whereareallthesickpeople ...,0.0000,Neutral


Saved compiled data to sentiment_data.csv
